# 법령 임베딩과 검색 확인

2026-11-10 Colab T4 실행 완료: **법령95개·청크165개 → 1024차원 벡터165개**. 유한값·L2정규화·동일입력 재인코딩·저장재개방·ZIP검사 통과.

이 파일은 출력 셀을 비워 둔 재현용입니다. 같은 코드의 실제 실행 결과는 별도 output 폴더에 저장했습니다. 모델 가중치는 Colab 런타임에서 받으며 PC 설치·유료API키·유료업그레이드가 필요하지 않습니다. 기본 제공T4가 없으면CPU로 실행할 수 있습니다.

**검색 관련성 검수는 미완입니다.** 일부 예시에서 간접적·부적절한 법령이1위에 나오는 것을 확인했습니다. 점수는 코사인 유사도이며 위험확률·법적정답이 아닙니다.

`런타임 > 모두 실행` 후 `3. 직접 검색해 보기`의 계약 문구 입력칸에 조항을 넣고 관련 법령 검색을 누르세요. DB적재와 판례 임베딩은 하지 않습니다.


In [ ]:
!pip -q install "transformers==4.57.6" "tokenizers==0.22.2" "sentencepiece>=0.2"


## 1. 코드·검증된 청크 복원
기존 법령 원문과 청킹 파일은 수정하지 않습니다. 판례는 이번 임베딩에 포함하지 않습니다.

In [ ]:
import base64
from pathlib import Path
WORK=Path("/content/law_embedding")
WORK.mkdir(parents=True,exist_ok=True)
(WORK/"law_embedding.py").write_bytes(base64.b64decode("IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJCR0UtTTMgZGVuc2UgQ0xTIGVtYmVkZGluZ3MsIGNvc2luZSByZXRyaWV2YWwsIGFuZCBsb2NhbCBleHBvcnQuIE5vIERCIHdyaXRlcy4iIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQganNvbgppbXBvcnQgdGltZQppbXBvcnQgemlwZmlsZQpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZXpvbmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsLCBBdXRvVG9rZW5pemVyCgpNT0RFTF9JRCA9ICJCQUFJL2JnZS1tMyIKTU9ERUxfUkVWSVNJT04gPSAiMzA2OWRlZjAzM2NlOTFkOTA3MjU4ZjlhODMwZTQ0MjYxMGRiZmUwYiIKCkVYQU1QTEVfUVVFUklFUyA9IFsKICAgICLqs4Tslb0g6riw6rCEIOykkSDrqqjrk6Ag7IiY66as67mE7JmAIOyImOyEoOu5hOuKlCDsnoTssKjsnbjsnbQg67aA64u07ZWc64ukLiIsCiAgICAi7IOI66Gc7Jq0IOyEuOyeheyekOqwgCDqtaztlbTsoLjslbwg67O07Kad6riI7J2EIOuPjOugpOykgOuLpOqzoCDtlanri4jri6QuIiwKICAgICLsm5TshLjrpbwg7ZWcIOuyiOunjCDsl7DssrTtlbTrj4Qg7KaJ7IucIOqzhOyVveydhCDtlbTsp4DtlZzri6QuIiwKICAgICLrrLXsi5zsoIHsnLzroZwg6rCx7Iug65CcIOqzhOyVveydhCDtlbTsp4DtlZjroKTrqbQg7Ja47KCcIO2aqOugpeydtCDrsJzsg53tlZjrgpjsmpQ/IiwKICAgICLsp5Hso7zsnbjsnbQg7Iuk7KCcIOqxsOyjvO2VnOuLpOuKlCDsnbTsnKDroZwg6rOE7JW96rCx7Iug7J2EIOqxsOygiO2WiOyKteuLiOuLpC4iLAogICAgIuyehOywqOyduOydgCDsoITsnoXsi6Dqs6Drpbwg7ZWY7KeAIOyViuq4sOuhnCDtlZzri6QuIiwKICAgICLth7TqsbDtlaAg65WMIOuPhOuwsOyZgCDsnqXtjJDsnYQg7KCE67aAIOyDiOqyg+ycvOuhnCDqtZDssrTtlbTslbwg7ZWc64ukLiIsCiAgICAi7J6E64yA7J247J20IO2VhOyalO2VoCDrlYzrp4jri6Qg7JuU7IS466W8IOydvOuwqeyggeycvOuhnCDsnbjsg4HtlZzri6QuIiwKICAgICLrs7Tspp3quIgg7J2867aA66W8IOyblOyEuOuhnCDrsJTqv4Ag65WMIOyggeyaqe2VmOuKlCDsoITtmZjsnKjsnYAg7Ja866eI7J246rCA7JqUPyIsCiAgICAi7IaM7JWh7J6E7LCo7J247J20IOuztOymneq4iCDspJEg7J287KCV7JWh7J2EIOyasOyEoOuzgOygnOuwm+uKlCDquIjslaEg6riw7KSA7J2EIOyVjOqzoCDsi7bsirXri4jri6QuIiwKICAgICLrs7Tspp3quIjsnYQg64+M66Ck67Cb7KeAIOuqu+2VnCDssYQg7J207IKs7ZW07JW8IO2VmOuKlOuNsCDsnoTssKjqtozrk7HquLDrpbwg7Iug7LKt7ZWY6rOgIOyLtuyKteuLiOuLpC4iLAogICAgIuq0gOumrOu5hOyXkCDsnqXquLDsiJjshKDstqnri7nquIjsnbQg7Y+s7ZWo65CY7Ja0IOyeiOuKlOuNsCDshLjsnoXsnpDqsIAg64yA7IugIOuCuCDrj4jsnYQg64+M66Ck67Cb7J2EIOyImCDsnojrgpjsmpQ/IiwKXQoKCmNsYXNzIERlbnNlTGF3U2VhcmNoOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNodW5rcyk6CiAgICAgICAgc2VsZi5jaHVua3MgPSBjaHVua3MKICAgICAgICBzZWxmLmRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgICAgIHNlbGYuZHR5cGUgPSB0b3JjaC5mbG9hdDE2IGlmIHNlbGYuZGV2aWNlLnR5cGUgPT0gImN1ZGEiIGVsc2UgdG9yY2guZmxvYXQzMgogICAgICAgIHNlbGYudG9rZW5pemVyID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoTU9ERUxfSUQsIHJldmlzaW9uPU1PREVMX1JFVklTSU9OLCB1c2VfZmFzdD1UcnVlKQogICAgICAgIGlmIG5vdCBzZWxmLnRva2VuaXplci5pc19mYXN0OgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImZhc3QgdG9rZW5pemVyIHJlcXVpcmVkIikKICAgICAgICBwcmludCgi66qo6424IOuLpOyatOuhnOuTnC/roZzrlKkg7Iuc7J6ROiIsIE1PREVMX0lELCAiZGV2aWNlOiIsIHNlbGYuZGV2aWNlLCBmbHVzaD1UcnVlKQogICAgICAgIHNlbGYubW9kZWwgPSBBdXRvTW9kZWwuZnJvbV9wcmV0cmFpbmVkKE1PREVMX0lELCByZXZpc2lvbj1NT0RFTF9SRVZJU0lPTiwgdG9yY2hfZHR5cGU9c2VsZi5kdHlwZSkKICAgICAgICBzZWxmLm1vZGVsLnRvKHNlbGYuZGV2aWNlKS5ldmFsKCkKICAgICAgICBpZiBpbnQoc2VsZi5tb2RlbC5jb25maWcuaGlkZGVuX3NpemUpICE9IDEwMjQ6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQkdFLU0zIOywqOybkCDrtojsnbzsuZgiKQogICAgICAgIHByaW50KCLrqqjrjbgg66Gc65SpIOyZhOujjCIsIGZsdXNoPVRydWUpCiAgICAgICAgc2VsZi52ZWN0b3JzID0gTm9uZQoKICAgIGRlZiBlbmNvZGUoc2VsZiwgdGV4dHMsIGJhdGNoX3NpemU9Tm9uZSk6CiAgICAgICAgYmF0Y2hfc2l6ZSA9IGJhdGNoX3NpemUgb3IgKDggaWYgc2VsZi5kZXZpY2UudHlwZSA9PSAiY3VkYSIgZWxzZSAyKQogICAgICAgIG91dHB1dCA9IFtdCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbih0ZXh0cyksIGJhdGNoX3NpemUpOgogICAgICAgICAgICBiYXRjaCA9IHRleHRzW3N0YXJ0OnN0YXJ0ICsgYmF0Y2hfc2l6ZV0KICAgICAgICAgICAgZW5jb2RlZCA9IHNlbGYudG9rZW5pemVyKGJhdGNoLCBwYWRkaW5nPVRydWUsIHRydW5jYXRpb249RmFsc2UsIHJldHVybl90ZW5zb3JzPSJwdCIpCiAgICAgICAgICAgIGlmIGVuY29kZWRbImlucHV0X2lkcyJdLnNoYXBlWzFdID4gODE5MjoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuuqqOuNuCDsnoXroKUg6ri47J20IOy0iOqzvDsg7J6Q64+ZIOygiOuLqO2VmOyngCDslYrsnYwiKQogICAgICAgICAgICBlbmNvZGVkID0ge2s6IHYudG8oc2VsZi5kZXZpY2UpIGZvciBrLCB2IGluIGVuY29kZWQuaXRlbXMoKX0KICAgICAgICAgICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogICAgICAgICAgICAgICAgaGlkZGVuID0gc2VsZi5tb2RlbCgqKmVuY29kZWQpLmxhc3RfaGlkZGVuX3N0YXRlCiAgICAgICAgICAgICAgICBkZW5zZSA9IEYubm9ybWFsaXplKGhpZGRlbls6LCAwLCA6XS5mbG9hdCgpLCBwPTIsIGRpbT0xKQogICAgICAgICAgICBvdXRwdXQuYXBwZW5kKGRlbnNlLmNwdSgpLm51bXB5KCkuYXN0eXBlKG5wLmZsb2F0MzIsIGNvcHk9RmFsc2UpKQogICAgICAgICAgICBpZiBsZW4odGV4dHMpID4gMTY6CiAgICAgICAgICAgICAgICBwcmludCgi7J6E67Kg65SpIOynhO2WiToiLCBtaW4oc3RhcnQgKyBiYXRjaF9zaXplLCBsZW4odGV4dHMpKSwgIi8iLCBsZW4odGV4dHMpLCBmbHVzaD1UcnVlKQogICAgICAgIHJldHVybiBucC5jb25jYXRlbmF0ZShvdXRwdXQsIGF4aXM9MCkKCiAgICBkZWYgc2VhcmNoKHNlbGYsIHF1ZXJ5LCB0b3Bfaz01KToKICAgICAgICBxdWVyeSA9IHF1ZXJ5LnN0cmlwKCkKICAgICAgICBpZiBub3QgcXVlcnk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuqygOyDie2VoCDqs4Tslb0g66y46rWs66W8IOyeheugpe2VmOyEuOyalCIpCiAgICAgICAgcW4gPSBsZW4oc2VsZi50b2tlbml6ZXIocXVlcnksIGFkZF9zcGVjaWFsX3Rva2Vucz1UcnVlLCB0cnVuY2F0aW9uPUZhbHNlKVsiaW5wdXRfaWRzIl0pCiAgICAgICAgaWYgcW4gPiA1MTI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuqygOyDiSDrrLjqtazqsIAgNTEy7Yag7YGw7J2EIOuEmOyKteuLiOuLpC4g7KGw7ZWtIO2VmOuCmOyUqSDsnoXroKXtlZjshLjsmpQiKQogICAgICAgIGlmIHNlbGYudmVjdG9ycyBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIuusuOyEnCDsnoTrsqDrlKnsnbQg7JWE7KeBIOyXhuyKteuLiOuLpCIpCiAgICAgICAgcXZlYyA9IHNlbGYuZW5jb2RlKFtxdWVyeV0pWzBdCiAgICAgICAgaWYgbm90IG5wLmlzZmluaXRlKHF2ZWMpLmFsbCgpIG9yIG5vdCBucC5pc2Nsb3NlKG5wLmxpbmFsZy5ub3JtKHF2ZWMpLCAxLjAsIGF0b2w9MWUtNSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuqygOyDieyWtCDrsqHthLAg7Jyg7ZWc6rCSL+ygleq3nO2ZlCDqsoDspp0g7Iuk7YyoIikKICAgICAgICBzY29yZXMgPSBzZWxmLnZlY3RvcnMgQCBxdmVjCiAgICAgICAgaWYgbm90IG5wLmlzZmluaXRlKHNjb3JlcykuYWxsKCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuqygOyDiSDsoJDsiJgg67mE7Jyg7ZWc6rCSIikKICAgICAgICBzZWxlY3RlZCwgc2VlbiA9IFtdLCBzZXQoKQogICAgICAgIGZvciBpZHggaW4gbnAuYXJnc29ydCgtc2NvcmVzKToKICAgICAgICAgICAgYyA9IHNlbGYuY2h1bmtzW2ludChpZHgpXQogICAgICAgICAgICBpZiBjWyJkb2N1bWVudF9rZXkiXSBpbiBzZWVuOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgc2Vlbi5hZGQoY1siZG9jdW1lbnRfa2V5Il0pCiAgICAgICAgICAgIHNlbGVjdGVkLmFwcGVuZCh7InJhbmsiOiBsZW4oc2VsZWN0ZWQpICsgMSwgInF1ZXJ5IjogcXVlcnksICJzY29yZSI6IGZsb2F0KHNjb3Jlc1tpZHhdKSwgImRvY3VtZW50X2tleSI6IGNbImRvY3VtZW50X2tleSJdLCAiY2h1bmtfa2V5IjogY1siY2h1bmtfa2V5Il0sICJsYXdfbmFtZSI6IGMuZ2V0KCJsYXdfbmFtZSIpLCAiYXJ0aWNsZSI6IGMuZ2V0KCJhcnRpY2xlIiksICJhcnRpY2xlX3RpdGxlIjogYy5nZXQoImFydGljbGVfdGl0bGUiKSwgInNvdXJjZV9lZmZlY3RpdmVfZGF0ZSI6IGMuZ2V0KCJzb3VyY2VfZWZmZWN0aXZlX2RhdGUiKSwgInNvdXJjZV91cmwiOiBjWyJzb3VyY2VfdXJsIl0sICJ0ZXh0IjogY1sidGV4dCJdLCAiY29udGV4dF9zcGFucyI6IGMuZ2V0KCJjb250ZXh0X3NwYW5zIiwgW10pLCAicmV2aWV3X2ZsYWdzIjogYy5nZXQoInJldmlld19mbGFncyIsIFtdKX0pCiAgICAgICAgICAgIGlmIGxlbihzZWxlY3RlZCkgPj0gdG9wX2s6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIHJldHVybiBzZWxlY3RlZAoKCmRlZiBidWlsZChpbnB1dF9maWxlLCBvdXRwdXRfZGlyKToKICAgIHQwID0gdGltZS50aW1lKCkKICAgIGlucHV0X2J5dGVzID0gUGF0aChpbnB1dF9maWxlKS5yZWFkX2J5dGVzKCkKICAgIGNodW5rcyA9IFtqc29uLmxvYWRzKGxpbmUpIGZvciBsaW5lIGluIGlucHV0X2J5dGVzLmRlY29kZSgidXRmLTgtc2lnIikuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KICAgIGlmIGxlbihjaHVua3MpICE9IDE2NSBvciBsZW4oe2NbImNodW5rX2tleSJdIGZvciBjIGluIGNodW5rc30pICE9IDE2NToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLtmITsnqwg6rOg7KCVIOyeheugpeydgCDspJHrs7Ug7JeG64qUIOyyre2BrDE2NeqwnOyXrOyVvCDtlanri4jri6QiKQogICAgaWYgbGVuKHtjWyJkb2N1bWVudF9rZXkiXSBmb3IgYyBpbiBjaHVua3N9KSAhPSA5NSBvciBub3QgYWxsKGNbImRvY3VtZW50X2tleSJdLnN0YXJ0c3dpdGgoIkxBV18iKSBmb3IgYyBpbiBjaHVua3MpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuuyleuguTk16rCcIOuylOychCDrtojsnbzsuZgiKQogICAgaWYgbm90IGFsbChjWyJ0b2tlbml6ZXJfcmV2aXNpb24iXSA9PSBNT0RFTF9SRVZJU0lPTiBmb3IgYyBpbiBjaHVua3MpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuyyre2CueqzvCDsnoTrsqDrlKkg7Yag7YGs64KY7J207KCAIO2MkOuzuOydtCDri6TrpoXri4jri6QiKQogICAgZW5naW5lID0gRGVuc2VMYXdTZWFyY2goY2h1bmtzKQogICAgdGV4dHMgPSBbY1siZW1iZWRkaW5nX3RleHQiXSBmb3IgYyBpbiBjaHVua3NdCiAgICB0b2tlbl9jb3VudHMgPSBbbGVuKGVuZ2luZS50b2tlbml6ZXIodCwgYWRkX3NwZWNpYWxfdG9rZW5zPVRydWUsIHRydW5jYXRpb249RmFsc2UpWyJpbnB1dF9pZHMiXSkgZm9yIHQgaW4gdGV4dHNdCiAgICBpZiB0b2tlbl9jb3VudHMgIT0gW2NbInRva2VuX2NvdW50Il0gZm9yIGMgaW4gY2h1bmtzXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLsi6TsoJwg7J6E67Kg65SpIOyeheugpSDthqDtgbAg7IiYIOu2iOydvOy5mCIpCiAgICBlbmdpbmUudmVjdG9ycyA9IGVuZ2luZS5lbmNvZGUodGV4dHMpCiAgICB2ZWN0b3JzID0gZW5naW5lLnZlY3RvcnMKICAgIGlmIHZlY3RvcnMuc2hhcGUgIT0gKDE2NSwgMTAyNCkgb3Igbm90IG5wLmlzZmluaXRlKHZlY3RvcnMpLmFsbCgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuuyoe2EsCDtmJXsg4Ev7Jyg7ZWc6rCSIOqygOymnSDsi6TtjKgiKQogICAgbm9ybXMgPSBucC5saW5hbGcubm9ybSh2ZWN0b3JzLCBheGlzPTEpCiAgICBpZiBub3QgbnAuYWxsY2xvc2Uobm9ybXMsIDEuMCwgYXRvbD0xZS01KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLrsqHthLAgTDIg7KCV6rec7ZmUIOyLpO2MqCIpCiAgICBpZiBucC5hbnkobnAubGluYWxnLm5vcm0odmVjdG9ycywgYXhpcz0xKSA9PSAwKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLsmIHrsqHthLAiKQogICAgcHJvYmUgPSBlbmdpbmUuZW5jb2RlKFt0ZXh0c1swXV0pWzBdCiAgICBwcm9iZV9zY29yZSA9IGZsb2F0KHByb2JlIEAgdmVjdG9yc1swXSkKICAgIGlmIG5vdCBucC5pc2Zpbml0ZShwcm9iZSkuYWxsKCkgb3Igbm90IG5wLmlzZmluaXRlKHByb2JlX3Njb3JlKSBvciBwcm9iZV9zY29yZSA8IDAuOTk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi6rCZ7J2AIOyeheugpSDsnqzsnbjsvZTrlKkg7J286rSA7ISxIOyLpO2MqCIpCiAgICBvdXRkaXIgPSBQYXRoKG91dHB1dF9kaXIpCiAgICBvdXRkaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgbnAuc2F2ZShvdXRkaXIgLyAi67KV66C5X+uyoe2EsC5ucHkiLCB2ZWN0b3JzLCBhbGxvd19waWNrbGU9RmFsc2UpCiAgICBpbmRleCA9IFt7InZlY3Rvcl9yb3ciOiBpLCAiY2h1bmtfa2V5IjogY1siY2h1bmtfa2V5Il0sICJkb2N1bWVudF9rZXkiOiBjWyJkb2N1bWVudF9rZXkiXSwgImxhd19uYW1lIjogYy5nZXQoImxhd19uYW1lIiksICJhcnRpY2xlIjogYy5nZXQoImFydGljbGUiKSwgInNvdXJjZV92ZXJzaW9uX3NlcSI6IGMuZ2V0KCJzb3VyY2VfdmVyc2lvbl9zZXEiKSwgInNvdXJjZV91cmwiOiBjWyJzb3VyY2VfdXJsIl19IGZvciBpLCBjIGluIGVudW1lcmF0ZShjaHVua3MpXQogICAgKG91dGRpciAvICLrspXroLlf67Kh7YSw7J24642x7IqkLmpzb25sIikud3JpdGVfdGV4dCgiIi5qb2luKGpzb24uZHVtcHMociwgZW5zdXJlX2FzY2lpPUZhbHNlKSArICJcbiIgZm9yIHIgaW4gaW5kZXgpLCBlbmNvZGluZz0idXRmLTgiKQogICAgd2l0aCAob3V0ZGlyIC8gIuuyleuguV/snoTrsqDrlKkuanNvbmwiKS5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOgogICAgICAgIGZvciBpLCBjIGluIGVudW1lcmF0ZShjaHVua3MpOgogICAgICAgICAgICByID0gZGljdChjKQogICAgICAgICAgICByLnVwZGF0ZSh7InZlY3Rvcl9yb3ciOiBpLCAiZW1iZWRkaW5nIjogdmVjdG9yc1tpXS50b2xpc3QoKSwgImVtYmVkZGluZ19zdGF0dXMiOiAiZ2VuZXJhdGVkIiwgImVtYmVkZGluZ19tb2RlbCI6IE1PREVMX0lELCAiZW1iZWRkaW5nX21vZGVsX3JldmlzaW9uIjogTU9ERUxfUkVWSVNJT04sICJlbWJlZGRpbmdfcG9vbGluZyI6ICJDTFNfbGFzdF9oaWRkZW5fc3RhdGUiLCAiZW1iZWRkaW5nX25vcm1hbGl6ZWQiOiBUcnVlLCAiZW1iZWRkaW5nX3N0b3JhZ2VfZHR5cGUiOiAiZmxvYXQzMiIsICJkYl9pZCI6IE5vbmUsICJkYl9kb2N1bWVudF9pZCI6IE5vbmV9KQogICAgICAgICAgICBoYW5kbGUud3JpdGUoanNvbi5kdW1wcyhyLCBlbnN1cmVfYXNjaWk9RmFsc2UpICsgIlxuIikKICAgIHJlc3VsdHMgPSBbciBmb3IgcSBpbiBFWEFNUExFX1FVRVJJRVMgZm9yIHIgaW4gZW5naW5lLnNlYXJjaChxLCA1KV0KICAgIGNvbHVtbnMgPSBbInF1ZXJ5IiwgInJhbmsiLCAic2NvcmUiLCAibGF3X25hbWUiLCAiYXJ0aWNsZSIsICJhcnRpY2xlX3RpdGxlIiwgInNvdXJjZV9lZmZlY3RpdmVfZGF0ZSIsICJzb3VyY2VfdXJsIiwgImRvY3VtZW50X2tleSIsICJjaHVua19rZXkiLCAidGV4dCIsICJyZXZpZXdfZmxhZ3MiXQogICAgd2l0aCAob3V0ZGlyIC8gIuuyleuguV/qsoDsg4nsmIjsi5wuY3N2Iikub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOC1zaWciLCBuZXdsaW5lPSIiKSBhcyBoYW5kbGU6CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoaGFuZGxlLCBmaWVsZG5hbWVzPWNvbHVtbnMpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICBmb3IgciBpbiByZXN1bHRzOgogICAgICAgICAgICB3cml0ZXIud3JpdGVyb3coe2s6ICI7Ii5qb2luKHJba10pIGlmIGsgPT0gInJldmlld19mbGFncyIgZWxzZSByW2tdIGZvciBrIGluIGNvbHVtbnN9KQogICAgKG91dGRpciAvICLrspXroLlf6rKA7IOJ7JiI7IucLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0cywgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZWFkYWJsZSA9IFsi67KV66C5IOydmOuvuOqygOyDiSDsmIjsi5wiLCAi7KCQ7IiY64qUIOy9lOyCrOyduCDsnKDsgqzrj4TsnbTrqbAg7JyE7ZeY7ZmV66WgL+uyleyggSDsoIHtlanshLEg7ZmV66Wg7J20IOyVhOuLmeuLiOuLpC4iLCAi7J6Q64+ZIOqygOyDiSDsiJzsnIQg7JiI7Iuc7J2066mwIOygleuLtSDqsoDsiJgv7KCV7ZmV64+EIO2PieqwgOulvCDrp4jsuZwg6rKw6rO86rCAIOyVhOuLmeuLiOuLpC4iLCAiIl0KICAgIGZvciBxIGluIEVYQU1QTEVfUVVFUklFUzoKICAgICAgICByZWFkYWJsZSArPSBbIj0iICogNjAsICLqsoDsg4kg66y46rWsOiAiICsgcV0KICAgICAgICBmb3IgciBpbiBbciBmb3IgciBpbiByZXN1bHRzIGlmIHJbInF1ZXJ5Il0gPT0gcV06CiAgICAgICAgICAgIHJlYWRhYmxlICs9IFsiIiwgc3RyKHJbInJhbmsiXSkgKyAiLiAiICsgc3RyKHJbImxhd19uYW1lIl0pICsgIiAiICsgc3RyKHJbImFydGljbGUiXSkgKyAiIC8g7Jyg7IKs64+EICIgKyBmb3JtYXQoclsic2NvcmUiXSwgIi40ZiIpLCAi7Lac7LKYOiAiICsgclsic291cmNlX3VybCJdLCByWyJ0ZXh0Il1dCiAgICAgICAgcmVhZGFibGUuYXBwZW5kKCIiKQogICAgKG91dGRpciAvICLrspXroLlf6rKA7IOJ7JiI7IucLnR4dCIpLndyaXRlX3RleHQoIlxuIi5qb2luKHJlYWRhYmxlKSwgZW5jb2Rpbmc9InV0Zi04LXNpZyIpCiAgICBtZXRhID0geyJjcmVhdGVkX2F0IjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCksICJzb3VyY2VfY2h1bmtzX3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KGlucHV0X2J5dGVzKS5oZXhkaWdlc3QoKSwgInNvdXJjZV9kb2N1bWVudHMiOiA5NSwgInZlY3Rvcl9jb3VudCI6IGxlbih2ZWN0b3JzKSwgInZlY3Rvcl9kaW1lbnNpb24iOiAxMDI0LCAidmVjdG9yX3NoYXBlIjogbGlzdCh2ZWN0b3JzLnNoYXBlKSwgInZlY3Rvcl9kdHlwZSI6IHN0cih2ZWN0b3JzLmR0eXBlKSwgImZpbml0ZV92YWx1ZXMiOiBib29sKG5wLmlzZmluaXRlKHZlY3RvcnMpLmFsbCgpKSwgInplcm9fdmVjdG9ycyI6IGludChucC5zdW0obm9ybXMgPT0gMCkpLCAibDJfbm9ybV9taW4iOiBmbG9hdChub3Jtcy5taW4oKSksICJsMl9ub3JtX21heCI6IGZsb2F0KG5vcm1zLm1heCgpKSwgInNhbWVfaW5wdXRfcmVlbmNvZGVfY29zaW5lIjogcHJvYmVfc2NvcmUsICJlbWJlZGRpbmdfbW9kZWwiOiBNT0RFTF9JRCwgInJldmlzaW9uIjogTU9ERUxfUkVWSVNJT04sICJwb29saW5nIjogIkNMU19sYXN0X2hpZGRlbl9zdGF0ZSIsICJub3JtYWxpemVkIjogVHJ1ZSwgImlucHV0X2ZpZWxkIjogImVtYmVkZGluZ190ZXh0IiwgIm1heF9pbnB1dF90b2tlbnMiOiBtYXgodG9rZW5fY291bnRzKSwgInRva2VuX2NvdW50c19tYXRjaF9jaHVua19hdWRpdCI6IFRydWUsICJkZXZpY2UiOiBzdHIoZW5naW5lLmRldmljZSksICJtb2RlbF9kdHlwZSI6IHN0cihlbmdpbmUuZHR5cGUpLCAidHJhbnNmb3JtZXJzX3ZlcnNpb24iOiBpbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbigidHJhbnNmb3JtZXJzIiksICJ0b3JjaF92ZXJzaW9uIjogdG9yY2guX192ZXJzaW9uX18sICJleGFtcGxlX3F1ZXJpZXMiOiBsZW4oRVhBTVBMRV9RVUVSSUVTKSwgImV4YW1wbGVfcmVzdWx0cyI6IGxlbihyZXN1bHRzKSwgInNlYXJjaF9zY29yZV9zZW1hbnRpY3MiOiAiY29zaW5lIHNpbWlsYXJpdHksIG5vdCByaXNrIHByb2JhYmlsaXR5IiwgInJldHJpZXZhbF9yZWxldmFuY2VfaHVtYW5fdmFsaWRhdGVkIjogRmFsc2UsICJsZWdhbF9hcHBsaWNhYmlsaXR5X3ZlcmlmaWVkIjogRmFsc2UsICJkYl9pbmdlc3RlZCI6IEZhbHNlLCAiZWxhcHNlZF9zZWNvbmRzIjogcm91bmQodGltZS50aW1lKCkgLSB0MCwgMil9CiAgICAob3V0ZGlyIC8gIuuyleuguV/snoTrsqDrlKnqsoDspp0uanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhtZXRhLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIGxvYWRlZCA9IG5wLmxvYWQob3V0ZGlyIC8gIuuyleuguV/rsqHthLAubnB5IiwgYWxsb3dfcGlja2xlPUZhbHNlKQogICAgaWYgbm90IG5wLmFycmF5X2VxdWFsKGxvYWRlZCwgdmVjdG9ycyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTlBZIOyggOyepSDsnqzqsJzrsKkg6rKA7KadIOyLpO2MqCIpCiAgICBuYW1lcyA9IFsi67KV66C5X+uyoe2EsC5ucHkiLCAi67KV66C5X+uyoe2EsOyduOuNseyKpC5qc29ubCIsICLrspXroLlf7J6E67Kg65SpLmpzb25sIiwgIuuyleuguV/qsoDsg4nsmIjsi5wuY3N2IiwgIuuyleuguV/qsoDsg4nsmIjsi5wuanNvbiIsICLrspXroLlf6rKA7IOJ7JiI7IucLnR4dCIsICLrspXroLlf7J6E67Kg65Sp6rKA7KadLmpzb24iXQogICAgYXJjaGl2ZSA9IG91dGRpci5wYXJlbnQgLyAi67KV66C5X+yehOuyoOuUqeqysOqzvC56aXAiCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShhcmNoaXZlLCAidyIsIGNvbXByZXNzaW9uPXppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyBwYWNrYWdlOgogICAgICAgIGZvciBuYW1lIGluIG5hbWVzOgogICAgICAgICAgICBwYWNrYWdlLndyaXRlKG91dGRpciAvIG5hbWUsIG5hbWUpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShhcmNoaXZlKSBhcyBwYWNrYWdlOgogICAgICAgIGlmIHBhY2thZ2UudGVzdHppcCgpIGlzIG5vdCBOb25lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJaSVAg6rKA7KadIOyLpO2MqCIpCiAgICBwcmludChqc29uLmR1bXBzKG1ldGEsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIGVuZ2luZSwgbWV0YSwgYXJjaGl2ZQoKCmRlZiBzaG93X3NlYXJjaChlbmdpbmUpOgogICAgaW1wb3J0IGlweXdpZGdldHMgYXMgd2lkZ2V0cwogICAgZnJvbSBJUHl0aG9uLmRpc3BsYXkgaW1wb3J0IGRpc3BsYXksIGNsZWFyX291dHB1dAogICAgdGV4dGJveCA9IHdpZGdldHMuVGV4dCh2YWx1ZT0i6rOE7JW9IOq4sOqwhCDspJEg66qo65OgIOyImOumrOu5hOuKlCDsnoTssKjsnbjsnbQg67aA64u07ZWc64ukLiIsIGRlc2NyaXB0aW9uPSLqs4Tslb0g66y46rWsOiIsIGxheW91dD13aWRnZXRzLkxheW91dCh3aWR0aD0iOTUlIikpCiAgICBidXR0b24gPSB3aWRnZXRzLkJ1dHRvbihkZXNjcmlwdGlvbj0i6rSA66CoIOuyleuguSDqsoDsg4kiLCBidXR0b25fc3R5bGU9InByaW1hcnkiKQogICAgb3V0cHV0ID0gd2lkZ2V0cy5PdXRwdXQoKQogICAgZGVmIG9uX2NsaWNrKF8pOgogICAgICAgIHdpdGggb3V0cHV0OgogICAgICAgICAgICBjbGVhcl9vdXRwdXQod2FpdD1UcnVlKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICByb3dzID0gZW5naW5lLnNlYXJjaCh0ZXh0Ym94LnZhbHVlLCA1KQogICAgICAgICAgICAgICAgcHJpbnQoIuygkOyImOuKlCDsnKDsgqzrj4TsnbTrqbAg7JyE7ZeY7ZmV66Wg7J20IOyVhOuLmeuLiOuLpC4g7KCB7Jqp7KGw6rG06rO8IOywuOyhsOyhsOusuOydgCDrs4Trj4TroZwg7ZmV7J247ZWY7IS47JqULlxuIikKICAgICAgICAgICAgICAgIGZvciByIGluIHJvd3M6CiAgICAgICAgICAgICAgICAgICAgcHJpbnQoc3RyKHJbInJhbmsiXSkgKyAiLiAiICsgc3RyKHJbImxhd19uYW1lIl0pICsgIiAiICsgc3RyKHJbImFydGljbGUiXSkgKyAiIC8g7Jyg7IKs64+EICIgKyBmb3JtYXQoclsic2NvcmUiXSwgIi40ZiIpKQogICAgICAgICAgICAgICAgICAgIHByaW50KHJbInRleHQiXSkKICAgICAgICAgICAgICAgICAgICBwcmludCgi6rO17IudIOy2nOyymDoiLCByWyJzb3VyY2VfdXJsIl0pCiAgICAgICAgICAgICAgICAgICAgcHJpbnQoKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgICAgIHByaW50KHR5cGUoZXhjKS5fX25hbWVfXyArICI6ICIgKyBzdHIoZXhjKSkKICAgIGJ1dHRvbi5vbl9jbGljayhvbl9jbGljaykKICAgIGRpc3BsYXkodGV4dGJveCwgYnV0dG9uLCBvdXRwdXQpCiAgICBvbl9jbGljayhOb25lKQogICAgcmV0dXJuIHRleHRib3gsIGJ1dHRvbiwgb3V0cHV0CgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0taW5wdXQiLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCBkZWZhdWx0PSJlbWJlZGRpbmdfb3V0cHV0IikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBidWlsZChhcmdzLmlucHV0LCBhcmdzLm91dHB1dCkK"))
(WORK/"law_chunks.jsonl").write_bytes(base64.b64decode("..."))
print("입력 준비 완료:",WORK)


## 2. 모델 로딩·임베딩·검증·검색 예시
165×1024, 유한값, 영벡터, L2 정규화, 같은 입력 재인코딩, 저장 재개방 및 ZIP 검사를 수행합니다. 문서행 순서는 별도 인덱스에 기록합니다. 예시 검색 결과는 수기 정답 평가가 아닙니다.

In [ ]:
import runpy
MODULE=runpy.run_path(str(WORK/"law_embedding.py"),run_name="law_embedding_module")
ENGINE,EMBED_AUDIT,EMBED_ZIP=MODULE["build"](WORK/"law_chunks.jsonl",WORK/"embedding_output")


## 3. 직접 검색해 보기
아래 입력칸에 계약서 조항 한 개를 넣고 `관련 법령 검색`을 누르세요. 상위5개 조문의 원문과 출처가 나옵니다. **수치가 높다고 위법·무효라는 뜻이 아닙니다.**

런타임 연결이 끊기면 위 실행 셀부터 다시 실행해야 새 검색을 할 수 있습니다.

In [ ]:
SEARCH_UI=MODULE["show_search"](ENGINE)


## 4. 결과 내려받기
search_examples.txt 또는 CSV는 모델을 다시 로딩하지 않고 읽을 수 있습니다. NPY와 벡터인덱스는 항상 함께 보관하세요. JSONL에는 청크와1024차원 벡터를 함께 저장했습니다. DB에는 아직 넣지 않았습니다.

In [ ]:
from google.colab import files
files.download(str(EMBED_ZIP))
